In [ ]:


## Packages ---

import numpy as np
import pandas as pd
import os
from pathlib import Path
import getpass
from tqdm import tqdm
import re
from datetime import date
pd.options.display.float_format = '{:.2f}'.format


## File paths ---

export=False

user = getpass.getuser()
path_users = Path.home()

path_sp = path_users / 'Sacramento Area Council of Governments' / 'Regional Monitoring and Reporting - Documents'
path_sp2 = path_sp / 'Products' / 'CERF' / 'We Prosper Together' / 'Transportation Measures'
path_git = path_users / 'Documents' / 'Projects' / 'Regional-Monitoring' / 'Indicator_Gen'
path_code    = path_git / 'Data' / 'Census'
path_config0 = path_git / 'config'
path_config  = path_code / 'config'

path_server = Path(r'\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data')
path_plots  = Path(r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring")
path_access = Path(r'I:\Projects\Josh\Regional Monitoring\Accessibility')



## User defined functions ---

path_func = path_config0 / 'Functions.py'

with path_func.open("r") as f:
    exec(f.read())



def process_access_1(df, geography, access, metric):

    df['Reference'] = '2022 LODES, 2023 ACS'
    df['Year'] = 2023

    if geography == 'MPO':
        df['Geography'] = 'ValleyVision Service Area'
    if geography == 'County':
        df['Geography'] = 'County'
        df['Geography'] = df['name'].copy()

    access_metrics = [f'transit_{access}', f'bike_{access}', f'walk_{access}', f'drive_{access}']
    df = df[['Reference', 'Geography', 'Year'] + access_metrics]
    
    df = df.melt(id_vars=['Reference', 'Geography', 'Year'], var_name='Mode', value_name=metric)

    conditions = [
        df['Mode'] == f'transit_{access}'
        , df['Mode'] == f'bike_{access}'
        , df['Mode'] == f'walk_{access}'
        , df['Mode'] == f'drive_{access}'
    ]
    choices = ['Public Transit', 'Bike', 'Walk', 'Drive']

    df['Mode'] = np.select(conditions, choices, default='no')

    return df


def process_access_2(df, geography, access, metric):

    df['Reference'] = '2024 overture, 2023 ACS'
    df['Year'] = 2023

    if geography == 'MPO':
        df['Geography'] = 'ValleyVision Service Area'
    if geography == 'County':
        df['Geography'] = df['name'].copy()

    access_metrics = [f'transit_{access}', f'bike_{access}', f'walk_{access}', f'drive_{access}']
    df = df[['Reference', 'Geography', 'Race_Ethnicity', 'Year'] + access_metrics]

    df = df.melt(id_vars=['Reference', 'Geography', 'Race_Ethnicity', 'Year'], var_name='Mode', value_name=metric)

    conditions = [
    df['Mode'] == f'transit_{access}'
    , df['Mode'] == f'bike_{access}'
    , df['Mode'] == f'walk_{access}'
    , df['Mode'] == f'drive_{access}'
    ]
    choices = ['Public Transit', 'Bike', 'Walk', 'Drive']
    df['Mode'] = np.select(conditions, choices, default='no')

    return df


def export_transit(df):
    df_about = write_about(sample_type    = sample_type
                           , indicator    = indicator
                           , year_start   = year_start
                           , year_end     = year_end
                           , path_config0 = path_config0
                           , geography    = geography)
    if file_out.is_file():
        with pd.ExcelWriter(file_out, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
            df_about.to_excel(writer, sheet_name='About'  , index=False, header=False)
            df      .to_excel(writer, sheet_name=geography, index=False,             )
    else:
        with pd.ExcelWriter(file_out, engine='xlsxwriter') as writer:
            df_about.to_excel(writer, sheet_name='About'  , index=False, header=False)
            df      .to_excel(writer, sheet_name=geography, index=False,             )

        

In [ ]:

year_start = 2023
year_end = 2023
sample_type = 'TMAG'


TravelAccess_1

In [ ]:

# Indicator
indicator = 'TravelAccess_1'
folder = 'Mode'
geography = 'MPO'



## Employment ---
metric = 'Number of Jobs'
access = 'emp'
file_in = path_access / f"tl_2020_valleyvision__access_pop_{access}.csv"
df_emp = pd.read_csv(file_in)
df_emp = process_access_1(df_emp, geography, access, metric)
df_emp = df_emp.dropna()
display(df_emp.head())



## Merge ---

if export:
    file_out = path_sp2 / f'{indicator} Valley Vision Service Area.xlsx'
    export_transit(df_emp)



In [ ]:



indicator = 'TravelAccess_1'
folder = 'Mode'
geography = 'County'


## Employment ---
metric = 'Number of Jobs'
access = 'emp'
file_in = path_access / f"tl_2020_valleyvision_county__access_pop_{access}.csv"
df_emp = pd.read_csv(file_in)
df_emp = process_access_1(df_emp, geography, access, metric)
df_emp = df_emp.dropna()
display(df_emp)



## Merge ---

df_emp['Sort'] = pd.Categorical(df_emp['Geography'], [
    'El Dorado'
    , 'Placer'
    , 'Sacramento'
    , 'Sutter'
    , 'Yolo'
    , 'Yuba'
])
df_emp = df_emp.sort_values(['Sort'], ascending=[True])
df_emp = df_emp.drop(['Sort'], axis=1)
df_emp = df_emp.reset_index(drop=True)


if export:
    file_out = path_sp2 / f'{indicator} {geography}.xlsx'
    export_transit(df_emp)




TravelAccess_2

In [ ]:


indicator = 'TravelAccess_2'
folder = 'Race'
geography = 'MPO'


## Employment ---

access = 'emp'
metric = 'Number of Jobs'

file_in1 = path_access / f"tl_2020_valleyvision__access_asian_{access}.csv"
file_in2 = path_access / f"tl_2020_valleyvision__access_black_{access}.csv"
file_in3 = path_access / f"tl_2020_valleyvision__access_white_{access}.csv"
file_in4 = path_access / f"tl_2020_valleyvision__access_hispanic_{access}.csv"

df_emp1 = pd.read_csv(file_in1)
df_emp2 = pd.read_csv(file_in2)
df_emp3 = pd.read_csv(file_in3)
df_emp4 = pd.read_csv(file_in4)

df_emp1['Race_Ethnicity'] = 'Asian (NH)'
df_emp2['Race_Ethnicity'] = 'Black or African American (NH)'
df_emp3['Race_Ethnicity'] = 'White (NH)'
df_emp4['Race_Ethnicity'] = 'Hispanic or Latino'

df_emp = pd.concat([df_emp1, df_emp2, df_emp3, df_emp4])

df_emp = process_access_2(df_emp, geography, access, metric)
display(df_emp.head())


## Merge ---


if export:
    file_out = path_sp2 / f'{indicator} Valley Vision Service Area.xlsx'
    export_transit(df_emp)



In [ ]:


indicator = 'TravelAccess_2'
folder = 'Race'
geography = 'County'


## Employment ---
access = 'emp'
metric = 'Number of Jobs'

file_in1 = path_access / f"tl_2020_valleyvision_county__access_asian_{access}.csv"
file_in2 = path_access / f"tl_2020_valleyvision_county__access_black_{access}.csv"
file_in3 = path_access / f"tl_2020_valleyvision_county__access_white_{access}.csv"
file_in4 = path_access / f"tl_2020_valleyvision_county__access_hispanic_{access}.csv"

df_emp1 = pd.read_csv(file_in1)
df_emp2 = pd.read_csv(file_in2)
df_emp3 = pd.read_csv(file_in3)
df_emp4 = pd.read_csv(file_in4)

df_emp1['Race_Ethnicity'] = 'Asian (NH)'
df_emp2['Race_Ethnicity'] = 'Black or African American (NH)'
df_emp3['Race_Ethnicity'] = 'White (NH)'
df_emp4['Race_Ethnicity'] = 'Hispanic or Latino'

df_emp = pd.concat([df_emp1, df_emp2, df_emp3, df_emp4])

df_emp = process_access_2(df_emp, geography, access, metric)
display(df_emp.head())


## Merge ---

if export:
    file_out = path_sp2 / f'{indicator} {geography}.xlsx'
    export_transit(df_emp)



